## Metadata Query Agent — Server-Side Batch Evaluation (Ground Truth)

This notebook evaluates the **deployed** Metadata Query Agent (AgentCore Runtime,
`MetadataQueryRuntimeArn`) **entirely server-side** with Amazon Bedrock **AgentCore Batch
Evaluations** (`bedrock_agentcore.evaluation.BatchEvaluationRunner`). Unlike the previous
revision, **no scoring runs locally** — the only thing this notebook does on the client is
*invoke the agent once per ground-truth row* (which is unavoidable: the agent has to run to
produce spans) and *read its response payload* to record cost/latency.

### Why batch (server-side) instead of the on-demand toolkit
The on-demand `Evaluation.run` path filters spans by parsing `cloud.resource_id` and was
observed to raise *"No spans found"* for this runtime. The **batch** service discovers spans
by **service name + session id + time range** (not `cloud.resource_id`), so it does not hit
that failure mode.

### Why SESSION-level evaluators (the multi-turn fix)
This agent can answer in **one** turn, or ask a **clarifying question** first and answer on a
later turn. A SESSION-level evaluator reads the whole conversation via `{context}` and scores
the **final** answer once — the right granularity for a multi-turn agent (a clarify-then-answer
conversation is judged on its destination, not on each intermediate turn).

> **Every turn now emits an answer span (fixed in the agent).** A Phase-2 clarification turn
> that makes no model/tool call used to produce only the structural `invoke_graph` /
> `POST /invocations` spans, so a TRACE-level evaluator raised
> `ValidationException: no spans to evaluate` and the batch failed the *whole* session (the
> historical `status=FAILED` runs). The agent now fixes this at the source —
> `shared/answer_span.emit_answer_span` writes a deterministic span (under the
> `strands.telemetry.tracer` scope the harvester reads) on the clarification AND final-answer
> paths, so clarify-only turns always have an evaluable span and the judge grades the text the
> user actually saw.

We still use **SESSION-level** custom judges (not TRACE) because answer quality for a
conversation is a once-per-conversation question and `{expected_response}` is TRACE-only — so
the old TRACE `QueryAnswerFaithfulness` is replaced with a SESSION `FinalAnswerFaithfulness`,
and the per-turn TRACE built-ins (`Builtin.Correctness`, `Builtin.Helpfulness`) are dropped.

### What gets scored (all server-side, all SESSION-level)
**Built-in evaluators**

| Evaluator | Level | Ground truth |
|:--|:--|:--|
| `Builtin.GoalSuccessRate` | Session | `assertions` (path + final-answer assertion) |

**Custom LLM-as-Judge evaluators** (created with `create_evaluator`, scored in-service — no Lambda)

| Evaluator | Level | Placeholders | Checks |
|:--|:--|:--|:--|
| `FinalAnswerFaithfulness` | SESSION | `{context}`, `{assertions}` | The conversation's FINAL answer matches the expected answer (threaded into `{assertions}`) |
| `SqlGrounded` | SESSION | `{context}`, `{available_tools}` | Every table/column in the executed SQL is present in the retrieved schema context — Phase 1 KB chunks / Phase 3 slice (no hallucinated schema) |
| `ToolCallOrdering` | SESSION | `{context}`, `{available_tools}` | Retrieved schema (KB chunks / slice) precedes the execute_sql_query call (graph ordering invariant) |

> **Placeholder constraint.** Per the AWS `CreateEvaluator` docs, a SESSION evaluator may
> reference only `{context}`, `{available_tools}`, `{assertions}`,
> `{expected_tool_trajectory}`, `{actual_tool_trajectory}`. `{expected_response}` is
> **TRACE-only** — so the expected final answer is threaded in as a SESSION assertion
> (`eval_multiturn.build_trajectory_assertions`, prefixed `FINAL_ANSWER_ASSERTION_PREFIX`)
> and `FinalAnswerFaithfulness` reads it from `{assertions}`.

### Metrics captured per query (from the agent's own response, not the evaluators)
The batch result exposes only the **evaluators'** token usage, never the **agent's**. So the
`agent_invoker` records, per turn, the query agent's `usage` (input/output/total tokens),
`runtimeMs`, and wall-clock latency into `AGENT_RUNS`.

### Prerequisites
- **Notebook 1** run to `%store metadata_id`
- **AgentCore stack** deployed; Metadata Query Agent running
- `bedrock-agentcore>=1.11.0` and `bedrock-agentcore-starter-toolkit==0.3.9`

In [ ]:
# Prereq (uncomment if not already installed):
# !pip install bedrock-agentcore-starter-toolkit==0.3.9

## 1. Setup & Credentials

In [ ]:
import os
import json
import uuid
import time
import boto3
import pandas as pd
from botocore.config import Config
from datetime import datetime, timezone
from dotenv import load_dotenv
from IPython.display import Markdown, display

# Load .env if present, then set defaults
load_dotenv(dotenv_path='.env', override=False)
os.environ.setdefault('AWS_PROFILE', 'huthmac+demo')
os.environ.setdefault('AWS_DEFAULT_REGION', 'us-east-1')

PROJECT_NAME = os.environ.get('PROJECT_NAME', 'semantic-layer')

config = Config(retries={'max_attempts': 10, 'mode': 'standard'}, signature_version='v4')

# Create session with AWS profile
session = boto3.Session(profile_name=os.environ['AWS_PROFILE'])
region = session.region_name or 'us-east-1'

# Verify credentials
sts = session.client('sts', region_name=region, config=config)
identity = sts.get_caller_identity()
print(f"✓ AWS Credentials Verified:")
print(f"  Profile: {os.environ['AWS_PROFILE']}")
print(f"  Account: ...{identity['Account'][-4:]}")
print(f"  User/Role: {identity['Arn'].split('/')[-1]}")
print(f"  Region: {region}")

# Render full cell text in displayed tables — never truncate question/message/SQL
# columns (the default max_colwidth=50 cut ground-truth questions mid-word).
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)


In [ ]:
import re

def _redact_account_ids(obj):
    """Recursively replace AWS account IDs embedded in ARN strings with XXXXXXXXXXXX."""
    if isinstance(obj, dict):
        return {k: _redact_account_ids(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_redact_account_ids(v) for v in obj]
    if isinstance(obj, str):
        return re.sub(r'(arn:[^:]*:[^:]*:[^:]*:)\d{12}:', r'\1XXXXXXXXXXXX:', obj)
    return obj


In [ ]:
# ── OAuth M2M runtime invoker ───────────────────────────────────────────────
# AgentCore runtimes use CUSTOM_JWT (Cognito M2M) inbound auth.
# The boto3 bedrock-agentcore client (SigV4) no longer works; use a Bearer
# token minted via Cognito client_credentials instead.
import base64
import urllib.parse as _urlparse
import urllib.request as _urlrequest

_BROWSER_UA = (
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
    'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36'
)
_OAUTH_SCOPE = 'semantic-layer-mcp/invoke'


def _get_m2m_creds() -> tuple:
    """Return (token_endpoint, client_id, client_secret) from CFN + Secrets Manager.

    Returns:
        Tuple of (token_endpoint str, client_id str, client_secret str).
    """
    cfn_client = session.client('cloudformation', region_name=region)
    auth_outputs = {}
    for stack_name in (f'{PROJECT_NAME}-auth', f'{PROJECT_NAME}-dev-auth'):
        try:
            stacks = cfn_client.describe_stacks(StackName=stack_name)['Stacks']
            auth_outputs = {o['OutputKey']: o['OutputValue'] for o in stacks[0].get('Outputs', [])}
            break
        except Exception:
            continue
    if not auth_outputs:
        raise RuntimeError(f'Auth stack not found for project {PROJECT_NAME}')
    token_endpoint = auth_outputs['McpHostedUiDomainUrl'] + '/oauth2/token'
    client_id = auth_outputs['M2mClientId']
    lam = session.client('lambda')
    cfg = lam.get_function_configuration(FunctionName=f'{PROJECT_NAME}-mcp-tools')
    secret_arn = cfg['Environment']['Variables']['M2M_CLIENT_SECRET_ARN']
    client_secret = session.client('secretsmanager').get_secret_value(
        SecretId=secret_arn,
    )['SecretString']
    return token_endpoint, client_id, client_secret


def _fetch_m2m_token() -> str:
    """Mint a Cognito client_credentials token for agent runtime invocations.

    Returns:
        OAuth access_token string.
    """
    token_endpoint, client_id, client_secret = _get_m2m_creds()
    body = _urlparse.urlencode({
        'grant_type': 'client_credentials',
        'scope': _OAUTH_SCOPE,
    }).encode()
    basic = base64.b64encode(f'{client_id}:{client_secret}'.encode()).decode('ascii')
    req = _urlrequest.Request(
        token_endpoint, data=body, method='POST',
        headers={
            'Content-Type': 'application/x-www-form-urlencoded',
            'Authorization': f'Basic {basic}',
            'User-Agent': _BROWSER_UA,
        },
    )
    with _urlrequest.urlopen(req, timeout=30) as resp:
        return json.loads(resp.read())['access_token']


def _invoke_runtime(arn: str, session_id: str, payload: bytes, *, qualifier: str = 'DEFAULT') -> bytes:
    """Invoke an AgentCore Runtime with Cognito Bearer auth.

    Parameters:
        arn: runtime ARN.
        session_id: X-Amzn-Bedrock-AgentCore-Runtime-Session-Id header value.
        payload: JSON-encoded request body.
        qualifier: runtime qualifier (default 'DEFAULT').
    Returns:
        Raw response body bytes.
    """
    token = _fetch_m2m_token()
    encoded_arn = arn.replace(':', '%3A').replace('/', '%2F')
    url = (
        f'https://bedrock-agentcore.{region}.amazonaws.com/'
        f'runtimes/{encoded_arn}/invocations?qualifier={qualifier}'
    )
    req = _urlrequest.Request(
        url, data=payload, method='POST',
        headers={
            'Authorization': f'Bearer {token}',
            'Content-Type': 'application/json',
            'X-Amzn-Bedrock-AgentCore-Runtime-Session-Id': session_id,
            'User-Agent': _BROWSER_UA,
        },
    )
    with _urlrequest.urlopen(req, timeout=900) as resp:
        return resp.read()


print('✓ OAuth M2M runtime invoker ready')

## 2. Load the Groundtruth Dataset

Schema (one record per row):

| Field | Type | Description |
|:------|:-----|:------------|
| `Natural_Language_Question` | string | The question fed to the agent |
| `Expected_Answer` | string | The natural-language answer the agent should produce |
| `Expected_SQL_Query` | string | The SQL query that would correctly answer the question |
| `Expected_SQL_Result` | list[dict] | The result rows that SQL should return |

The dataset drives the whole run: one scenario per row is built (section 5), the agent is
invoked once per scenario, and the groundtruth fields (`Expected_Answer` →
`expected_response`, the expected SQL/result → `assertions`) feed the server-side built-in
and custom evaluators.

In [ ]:
with open('../data/eval/groundtruth_dataset.json', 'r') as f:
    groundtruth = json.load(f)

# Data-query rows need SQL + expected result; advisory/meta rows (Expected_Tier
# == 'advisory') are answered by the intent-router advisory path and carry only
# a question + Expected_Answer, so they are exempt from the SQL requirement.
BASE_COLS = {'Natural_Language_Question', 'Expected_Answer'}
SQL_COLS = {'Expected_SQL_Query', 'Expected_SQL_Result'}
for i, row in enumerate(groundtruth):
    is_advisory = str(row.get('Expected_Tier', '')).lower() == 'advisory'
    required = BASE_COLS if is_advisory else (BASE_COLS | SQL_COLS)
    missing = required - set(row.keys())
    if missing:
        raise ValueError(f"Row {i} missing columns: {missing}")

# Slice for a quick pass; MAX_ROWS=0 (or unset) evaluates all rows.
# Read-only knob from the environment (see notebooks/.env) — never set here.
MAX_ROWS = int(os.environ.get('MAX_ROWS', '0'))
if MAX_ROWS > 0:
    groundtruth = groundtruth[:MAX_ROWS]
    print(f"⚠ MAX_ROWS={MAX_ROWS} — evaluating a {len(groundtruth)}-row slice")

df_gt = pd.DataFrame(groundtruth)

# ── Loud ground-truth quality check ──────────────────────────────────────────
# The server-side judges + built-in Correctness/GoalSuccessRate score the agent against
# Expected_Answer / Expected_SQL_Query / assertions. Rows whose expected fields are still
# 'PLACEHOLDER' will be scored against placeholder ground truth — the numbers are only
# meaningful once these are filled in. We surface the count instead of hiding it.
def _is_placeholder(v) -> bool:
    """True when a ground-truth field is empty or the literal 'PLACEHOLDER' sentinel."""
    return v in (None, '', [], 'PLACEHOLDER')

_ph_sql = sum(1 for r in groundtruth if _is_placeholder(r.get('Expected_SQL_Query')))
_ph_ans = sum(1 for r in groundtruth if _is_placeholder(r.get('Expected_Answer')))
print(f"✓ Loaded {len(df_gt)} groundtruth row(s)")
if _ph_sql or _ph_ans:
    print(f"  ⚠ {_ph_sql}/{len(groundtruth)} rows have a PLACEHOLDER Expected_SQL_Query and "
          f"{_ph_ans}/{len(groundtruth)} have a PLACEHOLDER Expected_Answer.")
    print(f"    Built-in Correctness / GoalSuccessRate and the custom judges will score "
          f"against placeholders for those rows — fill them in for meaningful scores.")
display(df_gt[['Natural_Language_Question', 'Expected_SQL_Query']])
# @multiturn-cell7  Multi-turn support: import the shared helpers (pure, no I/O)
# and report which rows are multi-turn so the run summary is honest.
try:
    from agents.shared.eval_multiturn import parse_multiturn_row
except ImportError:
    import sys
    sys.path.insert(0, '..')
    from agents.shared.eval_multiturn import parse_multiturn_row
_specs = [parse_multiturn_row(r, index=i) for i, r in enumerate(groundtruth)]
_mt = [s for s in _specs if len(s['turns']) > 1 or s['mode'] != 'scripted']
print(f"✓ {len(_specs)} scenario(s): {len(_mt)} multi-turn, {len(_specs) - len(_mt)} single-turn")


## 3. Resolve the AgentCore Runtime

Restore `metadata_id` (stored by notebook 1) and look up the Metadata Query Runtime
ARN from the AgentCore CloudFormation stack outputs.

In [ ]:
# ── Resolve the semantic-rag layer to evaluate ──
# Prefer the id stored by notebook 1 (%store), but VALIDATE it against the
# metadata table — a stored id from an earlier deploy can be stale (the layer
# was rebuilt under a new id), which makes the agent retrieve 0 KB sources and
# the batch FAIL. If the stored id is missing or no longer present/completed,
# fall back to the most-recently-updated COMPLETED semantic-rag layer.
_meta_table = session.resource('dynamodb', region_name=region).Table(
    os.environ.get('ONTOLOGY_METADATA_TABLE', f'{PROJECT_NAME}-metadata')
)


def _layer_ok(layer_id: str) -> bool:
    """True when ``layer_id`` exists in the metadata table with status=completed."""
    if not layer_id:
        return False
    resp = _meta_table.query(
        KeyConditionExpression='id = :i',
        ExpressionAttributeValues={':i': layer_id},
    )
    return any(it.get('status') == 'completed' for it in resp.get('Items', []))


def _latest_completed_semantic_rag() -> str:
    """Return the newest completed ``semantic-rag-*`` layer id (or '' if none)."""
    items, scan_kw = [], {
        'FilterExpression': 'contains(id, :p) AND #s = :c',
        'ExpressionAttributeNames': {'#s': 'status'},
        'ExpressionAttributeValues': {':p': 'semantic-rag', ':c': 'completed'},
    }
    while True:
        page = _meta_table.scan(**scan_kw)
        items.extend(page.get('Items', []))
        if 'LastEvaluatedKey' not in page:
            break
        scan_kw['ExclusiveStartKey'] = page['LastEvaluatedKey']
    if not items:
        return ''
    items.sort(key=lambda it: it.get('updatedAt') or it.get('createdAt') or '',
               reverse=True)
    return items[0]['id']


# Restore the stored id (no-op if notebook 1 hasn't run this session).
metadata_id = ''
try:
    %store -r metadata_id
except Exception:
    metadata_id = ''

EVAL_ID = metadata_id
if _layer_ok(EVAL_ID):
    print(f"✓ Using EVAL_ID from %store (validated completed): {EVAL_ID}")
else:
    fallback = _latest_completed_semantic_rag()
    if not fallback:
        raise RuntimeError(
            "No completed semantic-rag layer found in the metadata table. "
            "Build a Semantic RAG layer (notebook 1 / the admin UI) first."
        )
    print(f"⚠ Stored metadata_id ({EVAL_ID!r}) is stale/missing — falling back "
          f"to the latest completed semantic-rag layer.")
    EVAL_ID = fallback
    metadata_id = EVAL_ID
    # Refresh the store so downstream notebooks agree (the %store magic must be
    # its own statement — an inline comment after it is mis-parsed by IPython).
    %store metadata_id
    print(f"✓ Using EVAL_ID: {EVAL_ID}")

# ── Look up Metadata Query Runtime ARN + derive agent_id ──
cfn = session.client('cloudformation', region_name=region)
stack_name = f'{PROJECT_NAME}-agentcore'
outputs = cfn.describe_stacks(StackName=stack_name)['Stacks'][0]['Outputs']
output_map = {o['OutputKey']: o['OutputValue'] for o in outputs}

METADATA_QUERY_RUNTIME_ARN = output_map['MetadataQueryRuntimeArn']
# arn:aws:bedrock-agentcore:region:account:runtime/<agent_id>
agent_id = METADATA_QUERY_RUNTIME_ARN.rsplit('/', 1)[-1]

print(f"\n✓ AgentCore Runtime (from stack '{stack_name}'):")
print(f"  Runtime ARN: {METADATA_QUERY_RUNTIME_ARN}")
print(f"  Agent ID:    {agent_id}")

## 4. Create Custom (LLM-as-Judge) Evaluators

These run **inside AgentCore Evaluations** as LLM-as-a-Judge evaluators (the same pattern as the
AgentCore-evaluations workshop). All three are **SESSION-level** — each reads the **full session
conversation** via `{context}` (every turn, tool call, and tool result), so they tolerate the
agent's clarify-then-answer multi-turn flows. (Clarify turns now always emit an answer span
via shared/answer_span.emit_answer_span, and SESSION is the right granularity for judging a
multi-turn conversation's final answer; see the intro.)

| Evaluator | Level | Placeholders | Checks |
|:--|:--|:--|:--|
| `FinalAnswerFaithfulness` | SESSION | `{context}`, `{assertions}` | The conversation's FINAL answer matches the expected answer |
| `SqlGrounded` | SESSION | `{context}`, `{available_tools}` | Every table/column in the executed SQL appears in the retrieved schema context — KB chunks / slice (no hallucinated schema) |
| `ToolCallOrdering` | SESSION | `{context}`, `{available_tools}` | Retrieved schema (KB chunks / slice) precedes the execute_sql_query call (graph ordering invariant) |

> **How `FinalAnswerFaithfulness` gets the expected answer.** SESSION evaluators cannot use
> `{expected_response}` (TRACE-only). The expected answer is threaded into `{assertions}` as a
> line prefixed `The conversation's final answer matches: …` (by
> `eval_multiturn.build_trajectory_assertions`), and the judge extracts it from there.

> **How `SqlGrounded` sees the SQL and the schema without a Lambda.** The deployed agent runs a
> deterministic graph (phase fns), so the `{context}` placeholder (built from the session
> spans) carries the Phase 1 KB chunks / Phase 3 schema slice and the `execute_sql_query` tool
> **call arguments** (the SQL). The judge reads both from `{context}` — verified against the
> deployed runtime's spans. No code-based/Lambda evaluator is needed.

> Re-running this cell creates **new** evaluators (unique name suffix). Delete old ones via
> `delete_evaluator` if you iterate often.


In [ ]:
_cp = session.client('bedrock-agentcore-control', region_name=region)
_SUFFIX = uuid.uuid4().hex[:8]
JUDGE_MODEL_ID = 'global.anthropic.claude-sonnet-4-6'

_BINARY_SCALE = {
    'numerical': [
        {'value': 0.0, 'label': 'fail',
         'definition': 'Does not satisfy the criterion.'},
        {'value': 1.0, 'label': 'pass',
         'definition': 'Fully satisfies the criterion.'},
    ]
}


def _create_llm_judge(*, name: str, level: str, instructions: str) -> str:
    """Create a binary LLM-as-Judge evaluator and return its evaluatorId.

    Parameters:
        name: human-readable evaluator name (a unique suffix is appended).
        level: 'TRACE' or 'SESSION' — the granularity the service scores at.
        instructions: judge prompt; may reference ground-truth + context placeholders.

    Returns:
        The created evaluator's evaluatorId.
    """
    resp = _cp.create_evaluator(
        evaluatorName=f"{name}_{_SUFFIX}",
        level=level,
        evaluatorConfig={
            'llmAsAJudge': {
                'instructions': instructions,
                'ratingScale': _BINARY_SCALE,
                'modelConfig': {
                    'bedrockEvaluatorModelConfig': {
                        'modelId': JUDGE_MODEL_ID,
                        'inferenceConfig': {'maxTokens': 1024},
                    }
                },
            }
        },
    )
    return resp['evaluatorId']


# ── Why every custom judge is SESSION-level ──────────────────────────────────
# This agent answers in ONE turn OR asks a clarifying question first and answers on a LATER
# turn. SESSION judges read the whole conversation via {context} and score the FINAL answer
# once — the right granularity for a multi-turn agent.
# NOTE: clarify turns USED to emit no answer span (only invoke_graph + POST /invocations), so
# a TRACE evaluator raised "no spans to evaluate" and failed the whole session. That is now
# FIXED IN THE AGENT — shared/answer_span.emit_answer_span writes a deterministic span (under
# the strands.telemetry.tracer scope) on both the clarification and final-answer paths, so
# every turn has an evaluable span. We still use SESSION (not TRACE) because answer quality is
# a per-conversation question and {expected_response} is TRACE-only (we thread the expected
# answer into {assertions} instead). We drop the per-turn TRACE builtins
# (Builtin.Correctness, Builtin.Helpfulness) and replace TRACE QueryAnswerFaithfulness with a
# SESSION FinalAnswerFaithfulness.
#
# Placeholder constraint (AWS CreateEvaluator docs): a SESSION evaluator may reference ONLY
# {context}, {available_tools}, {assertions}, {expected_tool_trajectory},
# {actual_tool_trajectory}. {expected_response} is TRACE-ONLY. So the expected final answer
# is threaded in as a SESSION assertion (eval_multiturn.build_trajectory_assertions, prefixed
# FINAL_ANSWER_ASSERTION_PREFIX) and the faithfulness judge reads it from {assertions}.

# SESSION: final-answer faithfulness — the conversation's FINAL answer vs the expected answer
# (carried in {assertions} as "The conversation's final answer matches: ..."). Replaces the
# old TRACE QueryAnswerFaithfulness, which failed on every clarification turn AND on answer
# turns missing expected_response.
ANSWER_FAITHFUL_ID = _create_llm_judge(
    name='FinalAnswerFaithfulness',
    level='SESSION',
    instructions=(
        "You are a strict binary evaluator for a multi-turn text-to-SQL data agent.\n\n"
        "Session context (full conversation across ALL turns — user prompts, assistant "
        "responses, tool calls, and tool results):\n{context}\n\n"
        "Assertions about this session:\n{assertions}\n\n"
        "Exactly one assertion begins with 'The conversation's final answer matches:' — the "
        "text after that prefix is the EXPECTED final answer. The agent may take several "
        "turns (e.g. ask a clarifying question, get a reply, then answer); judge ONLY the "
        "FINAL substantive answer the agent gives at the end of the conversation.\n\n"
        "Score 1 (pass) iff that final answer is factually consistent with the expected "
        "answer — same key numbers, entities, and conclusion. Score 0 (fail) if it "
        "contradicts the expected answer, invents figures, omits the requested result, or if "
        "the conversation never reaches a substantive answer (e.g. ends still asking for "
        "clarification). If no assertion carries the expected-answer prefix, score 0 and note "
        "that the ground truth is missing. Briefly justify your score."
    ),
)

# SESSION: SQL grounding — every table/column in the executed SQL must appear in the retrieved
# schema context. The deployed agent runs a deterministic Tier 2 graph (phase fns), NOT a ReAct
# tool loop: retrieval/disambiguation are graph phases, so {context} carries the Phase 1 KB
# chunks / Phase 3 schema slice + the execute_sql_query tool CALL arguments (the SQL) — there is
# no retrieve_kb_context tool call. Server-side equivalent of the old local judge — no Lambda.
SQL_GROUNDED_ID = _create_llm_judge(
    name='SqlGrounded',
    level='SESSION',
    instructions=(
        "You are a strict binary grounding evaluator for a text-to-SQL data agent.\n\n"
        "Session context (full conversation, including tool calls and tool results):\n"
        "{context}\n\n"
        "Available tools: {available_tools}\n\n"
        "This agent runs a deterministic resolution graph: it retrieves Knowledge Base "
        "schema for the question, assembles a SCHEMA SLICE (the allowed tables/columns/joins), "
        "generates SQL against that slice, then executes it with the `execute_sql_query` tool. "
        "The retrieval/slice steps are graph phases, not model tool calls, so do NOT expect a "
        "`retrieve_kb_context` tool call. In the context, locate:\n"
        "  (a) the RETRIEVED SCHEMA CONTEXT — the KB chunks / schema slice describing the "
        "allowed tables, columns, and joins (the ONLY schema the agent may use); and\n"
        "  (b) the ARGUMENTS of the `execute_sql_query` tool — the SQL the agent actually ran.\n\n"
        "IMPORTANT — degraded runs: this agent has a grounding gate that REFUSES to "
        "execute SQL it cannot ground in the slice. If there is NO execute_sql_query tool "
        "call in the context (the agent degraded / declined rather than running ungrounded "
        "SQL), the grounding invariant was UPHELD — score 1 (pass) and note 'no SQL executed "
        "(grounded refusal)'. Do NOT treat generated-but-not-executed SQL appearing in "
        "reasoning/answer text as executed SQL; only the execute_sql_query tool-call "
        "arguments count as executed.\n"
        "Otherwise, score 1 (pass) iff EVERY table, column, and join referenced in the "
        "executed SQL appears in the retrieved schema context (case-insensitive; tolerate "
        "aliases, quoted vs unquoted identifiers, and SQL builtin functions such as "
        "COUNT/SUM/DATE_TRUNC — those are not schema). Score 0 (fail) if the executed SQL "
        "references any table or column that is absent from the retrieved schema context "
        "(hallucinated schema), or if SQL was executed but no retrieved schema context is "
        "present at all (grounding cannot be verified). Briefly name the first offending "
        "identifier when you fail it."
    ),
)

# SESSION: ordering invariant — did the agent ground its SQL in retrieved schema BEFORE
# executing it? The deployed agent runs a deterministic graph (phase fns), so the retrieval
# and disambiguation steps are graph phases (KB chunks / schema slice in {context}), not tool
# calls; the only real tool span is execute_sql_query. We judge that a retrieved schema
# context precedes the execute_sql_query call, using the full context.
TOOL_ORDERING_ID = _create_llm_judge(
    name='ToolCallOrdering',
    level='SESSION',
    instructions=(
        "You are a strict binary evaluator checking whether a text-to-SQL agent grounded its "
        "SQL in retrieved schema BEFORE executing it.\n\n"
        "This agent runs a deterministic resolution graph, not a free-form tool loop. The "
        "prescribed flow for a NEW question is, in this order:\n"
        "  1. retrieve Knowledge Base schema for the question (graph phase — appears as KB "
        "chunks in the context, not a model tool call);\n"
        "  2. assemble + disambiguate a schema SLICE of the allowed tables/columns (graph phase);\n"
        "  3. generate SQL against that slice, then call the `execute_sql_query` tool to run it.\n\n"
        "Available tools: {available_tools}\n"
        "Session context (phase outputs + the execute_sql_query call, chronological): {context}\n\n"
        "Determine whether the retrieved schema context (KB chunks / schema slice) and the SQL "
        "generated from it appear BEFORE the `execute_sql_query` call. Ignore tool RESULTS and "
        "any other tools; judge only this ordering invariant.\n\n"
        "Score 1 (pass) iff a retrieved schema context is present and precedes the first "
        "`execute_sql_query` call (a follow-up that reuses already-in-scope schema and skips "
        "fresh retrieval is acceptable). ALSO score 1 if the conversation never executes SQL "
        "(e.g. it ends in a clarifying question) — there is no ordering violation to find. "
        "Score 0 (fail) if `execute_sql_query` is invoked with no retrieved schema context "
        "anywhere before it, or if SQL is executed against schema that was never "
        "retrieved/assembled. Briefly explain the offending ordering when you fail it."
    ),
)

CUSTOM_EVALUATOR_IDS = [ANSWER_FAITHFUL_ID, SQL_GROUNDED_ID, TOOL_ORDERING_ID]
print("✓ Custom server-side evaluators created (all SESSION-level, LLM-as-Judge, no Lambda):")
print(f"  FinalAnswerFaithfulness (SESSION) : {ANSWER_FAITHFUL_ID}")
print(f"  SqlGrounded             (SESSION) : {SQL_GROUNDED_ID}")
print(f"  ToolCallOrdering        (SESSION) : {TOOL_ORDERING_ID}")


## 5. Build the Dataset (one scenario per row) + agent_invoker

**Per-row scenarios** — each ground-truth question is its own `PredefinedScenario` with its own
session, so we get **per-query** evaluator scores (via `fetch_evaluation_events`) and clean
**per-query** agent token/latency rows.

The `agent_invoker` is the only client-side work: it invokes the synchronous query agent once
per scenario and records the agent's own cost/latency (from `metadata.usage` / `runtimeMs` and
wall-clock) into `AGENT_RUNS`, keyed by the framework session id. Its return value
(`agent_output`) is the answer text the TRACE evaluators score.


In [ ]:
# @multiturn-cell13  Chat-stream invoker (Option A) + multi-turn dataset builder.
import json, time
from bedrock_agentcore.evaluation.runner.invoker_types import (
    AgentInvokerInput, AgentInvokerOutput)
from bedrock_agentcore.evaluation.runner.dataset_types import Dataset
try:
    from agents.shared.eval_multiturn import (
        parse_multiturn_row, build_chat_payload, build_scenarios,
        parse_chat_stream_sse, format_agent_output, run_key)
except ImportError:
    import sys; sys.path.insert(0, '..')
    from agents.shared.eval_multiturn import (
        parse_multiturn_row, build_chat_payload, build_scenarios,
        parse_chat_stream_sse, format_agent_output, run_key)

# Detect the simulation extra once; gate simulated scenarios on it.
try:
    import strands_evals  # noqa: F401
    SIMULATED_ENABLED = True
except ImportError:
    SIMULATED_ENABLED = False
print(f"simulated mode: {'ENABLED' if SIMULATED_ENABLED else 'DISABLED (scripted only)'}")

AGENT_RUNS = {}              # keyed by run_key(session_id, turn_idx)

def agent_invoker(invoker_input: AgentInvokerInput) -> AgentInvokerOutput:
    """Drive the agent CHAT-STREAM path so it reads+persists history itself.

    The SDK reuses one session_id across a scenario's turns; the chat entrypoint
    keys DDB history off the payload sessionId, so turn N+1 resolves turn N's
    clarification exactly as in production. We parse the SSE response, fold any
    clarification option labels into the judge-visible output, and record
    per-turn cost/latency keyed by (session_id, turn_idx).
    """
    sid = invoker_input.session_id
    # turn index derived from AGENT_RUNS so a notebook re-run (AGENT_RUNS reset above) is safe
    turn_idx = sum(1 for k in AGENT_RUNS if k.startswith(f"{sid}#turn"))
    message = (invoker_input.payload if isinstance(invoker_input.payload, str)
               else json.dumps(invoker_input.payload))
    payload = build_chat_payload(message=message, session_id=sid,
                                 ontology_id=EVAL_ID, turn_idx=turn_idx)
    start = time.time(); err = None; parsed = {}
    try:
        raw = _invoke_runtime(METADATA_QUERY_RUNTIME_ARN, sid,
                              json.dumps(payload).encode("utf-8"))
        parsed = parse_chat_stream_sse(raw.decode("utf-8", errors="replace"))
        err = parsed.get("error")
    except Exception as exc:  # noqa: BLE001
        err = str(exc); print(f"  ⚠ [{sid} t{turn_idx}] {err}")
    AGENT_RUNS[run_key(sid, turn_idx)] = {
        "scenario_session": sid, "turn_idx": turn_idx, "message": message,
        "answer": parsed.get("answer", ""), "agent_sql": parsed.get("sql", ""),
        "clarified": bool(parsed.get("clarification")),
        "rows": parsed.get("rows", []), "usage": parsed.get("usage", {}),
        "runtime_ms": parsed.get("runtime_ms"),
        # Provenance tier (Step 5) — read by the correct-tier assertion in cell 17.
        "provenance": parsed.get("provenance") or {},
        "wall_clock_s": round(time.time() - start, 2), "invoke_error": err,
    }
    state = "clarify" if parsed.get("clarification") else ("answer" if parsed.get("sql") else "none")
    print(f"  {'OK' if err is None else 'XX'} [{sid} t{turn_idx}] "
          f"{AGENT_RUNS[run_key(sid, turn_idx)]['wall_clock_s']}s . {state}")
    return AgentInvokerOutput(agent_output=format_agent_output(parsed) or (err or ""))

_specs = [parse_multiturn_row(r, index=i) for i, r in enumerate(groundtruth)]
dataset = Dataset(scenarios=build_scenarios(
    _specs, ontology_id=EVAL_ID, simulated_enabled=SIMULATED_ENABLED))
EXPECTED_TRAJECTORY = ["execute_sql_query"]
print(f"✓ Dataset: {len(dataset.scenarios)} scenario(s) "
      f"({sum(1 for s in _specs if len(s['turns'])>1)} multi-turn)")


## 6. Run the Batch Evaluation (server-side)

`run_dataset_evaluation` invokes the agent for each scenario (via `agent_invoker`), waits
`ingestion_delay_seconds` for CloudWatch span ingestion, submits a single `StartBatchEvaluation`
job mixing the **built-in** and **custom** evaluators, and polls to completion. All scoring
happens in-service. Span discovery is by **service name + session id + time range**.


In [ ]:
# @multiturn-cell15  k-run batch runner: SESSION-only evaluator set (clarification-safe).
# A single batch is a noisy draw (LLM-judge + agent stochasticity). We run the WHOLE batch
# EVAL_K times (default 3) and AVERAGE the per-evaluator aggregate scores in cell 17, so the
# headline numbers are robust. Each run clears + repopulates AGENT_RUNS via agent_invoker and
# snapshots that run's aggregate scores, per-query events, agent_runs, and cost/latency.
from bedrock_agentcore.evaluation.runner.batch.batch_evaluation_runner import (
    BatchEvaluationRunner,
)
from bedrock_agentcore.evaluation.runner.batch.batch_evaluation_models import (
    BatchEvaluationRunConfig, BatchEvaluatorConfig, CloudWatchDataSourceConfig,
)
try:
    from agents.shared.eval_multiturn import group_runs_by_session
except ImportError:
    import sys; sys.path.insert(0, '..')
    from agents.shared.eval_multiturn import group_runs_by_session

SERVICE_NAME = f"{agent_id}.DEFAULT"
RUNTIME_LOG_GROUP = f"/aws/bedrock-agentcore/runtimes/{agent_id}-DEFAULT"
SPANS_LOG_GROUP = "aws/spans"
print(f"SERVICE_NAME : {SERVICE_NAME}")
print(f"LOG GROUPS   : {SPANS_LOG_GROUP}, {RUNTIME_LOG_GROUP}")

batch_data_source = CloudWatchDataSourceConfig(
    service_names=[SERVICE_NAME],
    log_group_names=[SPANS_LOG_GROUP, RUNTIME_LOG_GROUP],
    # Multi-turn sessions emit more spans, and clarification turns ingest a beat later;
    # a longer wait gives short clarify turns time to ingest their answer span before scoring.
    ingestion_delay_seconds=420,
)

# ── SESSION-ONLY evaluator set ───────────────────────────────────────────────
# SESSION judges read the full conversation via {context} and score the final answer once.
# (Clarify turns now always emit an answer span via shared/answer_span.emit_answer_span, so the
# old "TRACE errors on clarify turns → whole session FAILED" failure no longer occurs; we still
# drop the per-turn TRACE builtins Builtin.Correctness/Helpfulness because answer quality is a
# per-conversation question.) Builtin.GoalSuccessRate is SESSION-level and stays.
#   - FinalAnswerFaithfulness  → answer quality (replaces the old TRACE QueryAnswerFaithfulness)
#   - SqlGrounded              → no hallucinated schema in executed SQL
#   - ToolCallOrdering         → retrieval precedes execution
ALL_EVALUATOR_IDS = [
    'Builtin.GoalSuccessRate',        # SESSION — assertions (path + final-answer)
    *CUSTOM_EVALUATOR_IDS,            # SESSION — FinalAnswerFaithfulness, SqlGrounded, ToolCallOrdering
]

# Map evaluator id → display name for the per-run aggregate scores. Builtins pass through
# (their id IS 'Builtin.GoalSuccessRate'); custom evaluators map their UUID to a clean name.
_EVAL_NAME = {ANSWER_FAITHFUL_ID: 'FinalAnswerFaithfulness',
              SQL_GROUNDED_ID: 'SqlGrounded',
              TOOL_ORDERING_ID: 'ToolCallOrdering'}

# k-run repetition knob (read-only, from notebooks/.env). EVAL_K=1 → single run (old behaviour).
EVAL_K = int(os.environ.get('EVAL_K', '3'))
batch_runner = BatchEvaluationRunner(region=region)


def _scenario_id_for_session(sess):
    """Map a framework session id (scenario_id + '-' + uuid) back to its scenario_id."""
    for s in _specs:
        if sess and sess.startswith(s['scenario_id'] + '-'):
            return s['scenario_id']
    return sess or ''


def _aggregate_scores(result) -> dict:
    """Pull the per-evaluator average scores out of a finished batch result.

    Parameters:
        result: the BatchEvaluationRunner result for one run.
    Returns:
        {display_name: average_score|None} over ALL_EVALUATOR_IDS.
    """
    scores = {}
    ev = result.evaluation_results
    if ev is not None:
        for es in (ev.evaluator_summaries or []):
            st = es.statistics
            name = _EVAL_NAME.get(es.evaluator_id, es.evaluator_id)
            scores[name] = (round(st.average_score, 4)
                            if st and st.average_score is not None else None)
    return scores


def _agent_perf_snapshot() -> dict:
    """Summarise the CURRENT AGENT_RUNS into one run's cost/latency totals."""
    lat = [r['wall_clock_s'] for r in AGENT_RUNS.values() if r.get('wall_clock_s') is not None]

    def _usum(key: str) -> int:
        return sum(int((r.get('usage') or {}).get(key) or 0) for r in AGENT_RUNS.values())

    return {'turns': len(AGENT_RUNS),
            'avg_wall_clock_s': round(sum(lat) / len(lat), 2) if lat else None,
            'input_tokens': _usum('inputTokens'), 'output_tokens': _usum('outputTokens'),
            'total_tokens': _usum('totalTokens')}


def _fetch_events(result) -> list:
    """Fetch this run's per-query evaluator events (output stream lags COMPLETED — retry)."""
    for _ in range(6):
        try:
            evs = batch_runner.fetch_evaluation_events(result)
            if evs:
                return evs
        except (LookupError, ValueError):
            pass
        time.sleep(20)
    return []


def _event_rows_for_run(result) -> list:
    """Flatten this run's evaluator events into per-(scenario, evaluator) score rows."""
    grouped = group_runs_by_session(AGENT_RUNS)

    def _f(e, key):
        return e[key] if key in e else (e.get('attributes', {}) or {}).get(key)

    rows = []
    for e in _fetch_events(result):
        sess = _f(e, 'session.id') or _f(e, 'gen_ai.session.id')
        turns = grouped.get(sess, [])
        name = _f(e, 'gen_ai.evaluation.name')
        rows.append({
            'scenario_id': _scenario_id_for_session(sess),
            'question': (turns[0]['message'] if turns else ''),
            'evaluator': _EVAL_NAME.get(name, name),
            'score': _f(e, 'gen_ai.evaluation.score.value'),
            'label': _f(e, 'gen_ai.evaluation.score.label'),
        })
    return rows


def _run_one_batch(run_idx: int) -> dict:
    """Run the SESSION-only batch once (fresh AGENT_RUNS); return a per-run snapshot dict."""
    AGENT_RUNS.clear()  # agent_invoker repopulates this per turn for THIS run
    cfg = BatchEvaluationRunConfig(
        batch_evaluation_name=f"query_gt_batch_r{run_idx}_{uuid.uuid4().hex[:8]}",
        description=f"Server-side SESSION-only ground-truth eval of the metadata query agent "
                    f"(clarification-safe), run {run_idx}/{EVAL_K}.",
        evaluator_config=BatchEvaluatorConfig(evaluator_ids=ALL_EVALUATOR_IDS),
        data_source=batch_data_source,
        max_concurrent_scenarios=3,   # synchronous agent, up to 900s/row
        polling_timeout_seconds=3600,
        polling_interval_seconds=30,
    )
    print(f"\n=== Run {run_idx}/{EVAL_K}: {cfg.batch_evaluation_name} ===")
    res = batch_runner.run_dataset_evaluation(
        config=cfg, dataset=dataset, agent_invoker=agent_invoker)
    print(f"  ✓ status: {res.status}")
    if res.agent_invocation_failures:
        print(f"  ⚠ Agent invocation failures: {len(res.agent_invocation_failures)}")
    snap = {'run': run_idx, 'batch_id': res.batch_evaluation_id,
            'batch_arn': res.batch_evaluation_arn, 'status': str(res.status),
            'scores': _aggregate_scores(res), 'agent_perf': _agent_perf_snapshot(),
            'events': _event_rows_for_run(res),
            'agent_runs': list(AGENT_RUNS.values())}
    snap['_result'] = res  # kept in-memory only (not serialised) for the last-run detail cell
    print(f"  run {run_idx} scores: {snap['scores']}")
    return snap


print(f"\nEvaluators : {ALL_EVALUATOR_IDS}  (all SESSION-level)")
print(f"Scenarios  : {len(dataset.scenarios)}   ·   EVAL_K = {EVAL_K}")
print("Starting k-run batch evaluation (invokes the agent per row, then evaluates server-side)...")

RUNS = []           # per-run snapshots (scores + perf + events + agent_runs); averaged in cell 17
for _i in range(1, EVAL_K + 1):
    RUNS.append(_run_one_batch(_i))
    batch_result = RUNS[-1]['_result']  # keep the LAST run bound for the per-query detail cell

print(f"\n✓ Completed {EVAL_K} run(s)")


## 7. Results — aggregate scores, per-query detail, agent cost/latency

Three layers:
1. **Aggregate per-evaluator scores** — from `batch_result.evaluation_results` (in-service).
2. **Per-query evaluator scores** — `fetch_evaluation_events()` reads one event per
   turn-per-evaluator (score, label, explanation) from the batch output log stream, joined
   back to each question by session id.
3. **Per-query agent cost/latency** — from `AGENT_RUNS` (the agent's own tokens / runtimeMs /
   wall-clock; the batch result never carries these).


In [ ]:
# @multiturn-cell17  k-run mean scores + last-run per-query detail + preserved JSON dumps.
os.makedirs('../data/eval/results', exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# ════════════════════════════════════════════════════════════════════════════
# A. k-run MEAN aggregate scores (the headline) — averaged over RUNS from cell 15.
# ════════════════════════════════════════════════════════════════════════════
import statistics as _stats

_EVAL_ORDER = ['Builtin.GoalSuccessRate', 'FinalAnswerFaithfulness',
               'SqlGrounded', 'ToolCallOrdering']


def _mean_std(values: list):
    """Return (mean, std, n) over the non-None numeric values (std=0.0 when n<2)."""
    nums = [v for v in values if isinstance(v, (int, float))]
    if not nums:
        return None, None, 0
    mean = round(sum(nums) / len(nums), 4)
    std = round(_stats.pstdev(nums), 4) if len(nums) > 1 else 0.0
    return mean, std, len(nums)


# Per-evaluator mean ± std across the k runs.
mean_rows = []
for ev_name in _EVAL_ORDER:
    per_run = [r['scores'].get(ev_name) for r in RUNS]
    mean, std, n = _mean_std(per_run)
    mean_rows.append({'Evaluator': ev_name, 'MeanScore': mean, 'Std': std,
                      'Runs': n, 'PerRun': [r['scores'].get(ev_name) for r in RUNS]})
df_mean = pd.DataFrame(mean_rows)

# Mean agent cost/latency across runs.
def _perf_mean(key: str):
    return _mean_std([r['agent_perf'].get(key) for r in RUNS])[0]

agent_perf_mean = {
    'avg_wall_clock_s': _perf_mean('avg_wall_clock_s'),
    'input_tokens': _perf_mean('input_tokens'),
    'output_tokens': _perf_mean('output_tokens'),
    'total_tokens': _perf_mean('total_tokens'),
}

# Per-scenario mean GoalSuccess across runs (handy for downstream matched-subset analysis).
_per_scn = {}
for r in RUNS:
    for e in r['events']:
        if e['evaluator'] == 'Builtin.GoalSuccessRate' and isinstance(e['score'], (int, float)):
            _per_scn.setdefault(e['scenario_id'], []).append(e['score'])
per_scenario_goal_mean = {sid: round(sum(v) / len(v), 4) for sid, v in _per_scn.items()}

# Per-scenario agent detail across all k runs (emitted SQL / mean tokens / mean latency).
# Downstream notebook 9 reads this to build its apples-to-apples matched subset (rows where
# BOTH arms emitted SQL) for this arm WITHOUT re-running this notebook's agent. We key off the
# per-run agent_runs snapshots captured in cell 15, folding multi-turn sessions to their FINAL
# turn (the answer-bearing turn) and averaging tokens/latency over the runs that produced them.
def _scn_id_from_session(sess):
    for s in _specs:
        if sess and sess.startswith(s['scenario_id'] + '-'):
            return s['scenario_id']
    return sess or ''
_psa = {}
for r in RUNS:
    by_sess = {}
    for rec in r['agent_runs']:
        by_sess.setdefault(rec['scenario_session'], []).append(rec)
    for sess, recs in by_sess.items():
        recs.sort(key=lambda x: x.get('turn_idx', 0))
        final = recs[-1]
        sid = _scn_id_from_session(sess)
        d = _psa.setdefault(sid, {'sql_runs': 0, 'tokens': [], 'latency': []})
        if final.get('agent_sql'):
            d['sql_runs'] += 1
        d['tokens'].append(int((final.get('usage') or {}).get('totalTokens') or 0))
        d['latency'].append(sum(x.get('wall_clock_s') or 0 for x in recs))
# question text per scenario (the robust cross-notebook join key — gt-row-NN indices differ
# between notebooks because some filter the dataset, but the question text is stable).
_q_by_sid = {s['scenario_id']: (s['turns'][0].get('input', '') if s.get('turns') else '')
             for s in _specs}
per_scenario_agent = {
    sid: {'question': _q_by_sid.get(sid, ''),
          'emitted_sql': d['sql_runs'] > 0,        # emitted SQL in at least one run
          'sql_run_fraction': round(d['sql_runs'] / len(d['tokens']), 4) if d['tokens'] else 0.0,
          'avg_total_tokens': round(sum(d['tokens']) / len(d['tokens'])) if d['tokens'] else 0,
          'avg_wall_clock_s': round(sum(d['latency']) / len(d['latency']), 2) if d['latency'] else None}
    for sid, d in _psa.items()
}


print(f"=== k-run MEAN scores over EVAL_K={EVAL_K} run(s) ===")
display(df_mean)
print(f"\nMean agent cost/latency: avg_wall={agent_perf_mean['avg_wall_clock_s']}s  "
      f"total_tokens={agent_perf_mean['total_tokens']}")

# ── Persist the k-run mean summary (the file downstream notebooks 9 & 10 read). ──
kmean_file = f"../data/eval/results/metadata_query_kmean_eval_{timestamp}.json"
kmean = {
    'notebook': '2_metadata_query',
    'arm_label': 'normalized-s3',          # SemanticRAG over the normalized S3 layer
    'agent': 'metadata_query_agent',
    'agent_id': agent_id,
    'runtime_arn': METADATA_QUERY_RUNTIME_ARN,
    'eval_id': EVAL_ID,
    'eval_k': EVAL_K,
    'evaluator_ids': ALL_EVALUATOR_IDS,
    'custom_evaluators': {'FinalAnswerFaithfulness': ANSWER_FAITHFUL_ID,
                          'SqlGrounded': SQL_GROUNDED_ID, 'ToolCallOrdering': TOOL_ORDERING_ID},
    # Mean ± std per evaluator (the headline), plus the per-run scores that produced them.
    'mean_scores': {row['Evaluator']: row['MeanScore'] for row in mean_rows},
    'std_scores': {row['Evaluator']: row['Std'] for row in mean_rows},
    'per_run_scores': [{'run': r['run'], 'batch_id': r['batch_id'],
                        'status': r['status'], 'scores': r['scores'],
                        'agent_perf': r['agent_perf']} for r in RUNS],
    'agent_perf_mean': agent_perf_mean,
    'per_scenario_goal_mean': per_scenario_goal_mean,
    'per_scenario_agent': per_scenario_agent,
}
kmean = _redact_account_ids(kmean)
with open(kmean_file, 'w') as f:
    json.dump(kmean, f, indent=2, default=str)
print(f"\n✓ Wrote k-run mean summary → {kmean_file}")

# ════════════════════════════════════════════════════════════════════════════
# B. Per-query DETAIL for the LAST run (batch_result + AGENT_RUNS point at run k).
#    Preserves the metadata_query_batch_eval_*.json file notebook 4 globs for.
# ════════════════════════════════════════════════════════════════════════════
try:
    from agents.shared.eval_multiturn import group_runs_by_session
except ImportError:
    import sys; sys.path.insert(0, '..')
    from agents.shared.eval_multiturn import group_runs_by_session

print(f"\n── Last-run detail — Batch ID: {batch_result.batch_evaluation_id} "
      f"(status {batch_result.status}) ──")

# ── 1. Aggregate per-evaluator scores (last run) ─────────────────────────────
agg_rows = []
ev = batch_result.evaluation_results
if ev is not None:
    print(f"Sessions: completed={ev.number_of_sessions_completed} "
          f"failed={ev.number_of_sessions_failed} total={ev.total_number_of_sessions}")
    for es in (ev.evaluator_summaries or []):
        stats = es.statistics
        avg = (f"{stats.average_score:.3f}"
               if stats and stats.average_score is not None else None)
        agg_rows.append({'Evaluator': es.evaluator_id or 'unknown', 'AvgScore': avg,
                         'Evaluated': es.total_evaluated or 0, 'Failed': es.total_failed or 0})
else:
    print("⚠ No aggregate evaluation_results — check job status / spans.")
df_agg = pd.DataFrame(agg_rows)

grouped = group_runs_by_session(AGENT_RUNS)

# ── 2. Per-query evaluator events (last run) — reuse the helper from cell 15 ──
event_rows = _event_rows_for_run(batch_result)
# Re-add the explanation field for the displayed detail table.
def _f(e, key):
    return e[key] if key in e else (e.get('attributes', {}) or {}).get(key)
_explan = {}
for e in _fetch_events(batch_result):
    sess = _f(e, 'session.id') or _f(e, 'gen_ai.session.id')
    name = _f(e, 'gen_ai.evaluation.name')
    _explan[(_scenario_id_for_session(sess), _EVAL_NAME.get(name, name))] = \
        (str(_f(e, 'gen_ai.evaluation.explanation') or ''))[:400]
for row in event_rows:
    row['explanation'] = _explan.get((row['scenario_id'], row['evaluator']), '')
df_events = pd.DataFrame(event_rows)
print(f"Per-query evaluator events (last run): {len(df_events)}")

# ── 3. Per-turn trajectory table + clarification summary (last run) ──────────
turn_rows = []
for sess, turns in grouped.items():
    sid = _scenario_id_for_session(sess)
    for t in turns:
        turn_rows.append({
            'scenario_id': sid, 'turn': t.get('turn_idx'),
            'message': (t.get('message') or ''), 'clarified': t.get('clarified'),
            'has_sql': bool(t.get('agent_sql')), 'wall_s': t.get('wall_clock_s'),
            'error': t.get('invoke_error'),
        })
df_agent = pd.DataFrame(turn_rows)
if not df_agent.empty:
    df_agent = df_agent.sort_values(['scenario_id', 'turn']).reset_index(drop=True)

print("\n── Trajectory summary (last run, per scenario) ──")
for sess, turns in grouped.items():
    sid = _scenario_id_for_session(sess)
    clar = [t.get('turn_idx') for t in turns if t.get('clarified')]
    final_sql = bool(turns[-1].get('agent_sql')) if turns else False
    print(f"  {sid}: {len(turns)} turn(s) · clarified_on={clar or 'none'} · "
          f"re_clarified={len(clar) > 1} · final_sql={final_sql}")

# ── 3b. Correct-tier assertion (Step 5: intent router + provenance) ──────────
_spec_by_id = {s['scenario_id']: s for s in _specs}
tier_rows = []
for sess, turns in grouped.items():
    sid = _scenario_id_for_session(sess)
    spec = _spec_by_id.get(sid, {})
    expected = spec.get('expected_tier', 'semantic_sql')
    final = turns[-1] if turns else {}
    prov = final.get('provenance') or {}
    actual = prov.get('tier') or ('semantic_sql' if final.get('agent_sql') else 'unknown')
    tier_rows.append({
        'scenario_id': sid, 'question': (turns[0]['message'] if turns else '')[:70],
        'expected_tier': expected, 'actual_tier': actual, 'match': expected == actual,
    })
df_tier = pd.DataFrame(tier_rows)
_tier_correct = sum(1 for r in tier_rows if r['match'])
_tier_total = len(tier_rows)
print(f"\n── Correct-tier routing (last run): {_tier_correct}/{_tier_total} "
      f"({100*_tier_correct//_tier_total if _tier_total else 0}%) ──")
for r in tier_rows:
    print(f"  {'OK' if r['match'] else 'XX'} {r['scenario_id']}: "
          f"expected={r['expected_tier']} actual={r['actual_tier']}")
if not df_tier.empty:
    display(df_tier)

# ── 4. Agent cost/latency (last run) ─────────────────────────────────────────
_lat = [r['wall_clock_s'] for r in AGENT_RUNS.values() if r.get('wall_clock_s') is not None]
def _usage_sum(key):
    return sum(int((r.get('usage') or {}).get(key) or 0) for r in AGENT_RUNS.values())
agent_perf = {
    'turns': len(AGENT_RUNS), 'sessions': len(grouped),
    'avg_wall_clock_s': round(sum(_lat) / len(_lat), 2) if _lat else None,
    'input_tokens': _usage_sum('inputTokens'), 'output_tokens': _usage_sum('outputTokens'),
    'total_tokens': _usage_sum('totalTokens'),
}
print(f"\n── Query agent cost/latency (last run) ──")
print(f"  Avg wall-clock : {agent_perf['avg_wall_clock_s']}s  ·  tokens: {agent_perf['total_tokens']}")

# ── Persist the last-run detail (filename pattern + keys notebook 4 reads). ──
combined_file = f"../data/eval/results/metadata_query_batch_eval_{timestamp}.json"
combined = {
    'agent_id': agent_id, 'runtime_arn': METADATA_QUERY_RUNTIME_ARN, 'eval_id': EVAL_ID,
    'eval_k': EVAL_K, 'kmean_file': os.path.basename(kmean_file),
    'batch_evaluation_id': batch_result.batch_evaluation_id,
    'batch_evaluation_arn': batch_result.batch_evaluation_arn,
    'status': str(batch_result.status), 'evaluator_ids': ALL_EVALUATOR_IDS,
    'custom_evaluators': {'FinalAnswerFaithfulness': ANSWER_FAITHFUL_ID,
                          'SqlGrounded': SQL_GROUNDED_ID, 'ToolCallOrdering': TOOL_ORDERING_ID},
    'aggregate_summaries': agg_rows, 'per_query_events': event_rows,
    'agent_runs': list(AGENT_RUNS.values()), 'agent_performance': agent_perf,
    'correct_tier': {'rows': tier_rows, 'correct': _tier_correct, 'total': _tier_total},
}
combined = _redact_account_ids(combined)
with open(combined_file, 'w') as f:
    json.dump(combined, f, indent=2, default=str)
print(f"✓ Wrote last-run detail → {combined_file}")

print("\n=== Aggregate per-evaluator scores (last run) ===")
display(df_agg)
print("=== Per-query evaluator scores (last run) ===")
display(df_events)
print("=== Per-turn trajectory (last run) ===")
display(df_agent)


## 8. Store IDs for Downstream Notebooks

Save the batch evaluation id, the per-query session ids, and the agent id so downstream
notebooks (e.g. notebook 3, online evaluation) can reference this run.

In [ ]:
query_batch_id = batch_result.batch_evaluation_id
query_eval_session_ids = list(AGENT_RUNS.keys())
ondemand_agent_id = agent_id  # kept name for downstream notebook compatibility
%store query_batch_id
%store query_eval_session_ids
%store ondemand_agent_id

print("✓ Stored for downstream notebooks:")
print(f"  query_batch_id        = {query_batch_id}")
print(f"  query_eval_session_ids= {len(query_eval_session_ids)} session(s)")
print(f"  ondemand_agent_id     = {ondemand_agent_id}")
print(f"\n  Combined results: {combined_file}")


## Summary

This notebook evaluates the deployed Metadata Query Agent **entirely server-side** via
AgentCore Batch Evaluations. The only client-side work is invoking the agent once per
ground-truth turn (required to produce spans) and reading its response for cost/latency.

### Pipeline
1. **Custom evaluators** (LLM-as-Judge, no Lambda) — all **SESSION-level**:
   `FinalAnswerFaithfulness`, `SqlGrounded`, and `ToolCallOrdering`, created with
   `create_evaluator` and scored in-service.
2. **Dataset** — one `PredefinedScenario` per ground-truth row (per-conversation sessions);
   multi-turn rows carry several `Turn`s. `expected_response` is set on the **final** turn
   only (TRACE ground truth), and the expected answer is **also** threaded into `assertions`
   so the SESSION `FinalAnswerFaithfulness` judge can read it. Each scenario carries
   `expected_trajectory` and `assertions`.
3. **agent_invoker** — drives the chat-stream path (reads + persists DDB history) so turn N+1
   resolves turn N's clarification; records the agent's own tokens / `runtimeMs` / wall-clock
   per turn into `AGENT_RUNS`.
4. **BatchEvaluationRunner** — one `StartBatchEvaluation` job over the SESSION-only evaluator
   set.
5. **Results** — aggregate per-evaluator scores, **per-query** scores via
   `fetch_evaluation_events`, and **per-turn** agent cost/latency + trajectory.

### Why SESSION-level evaluators (the multi-turn fix)
This agent is multi-turn: it may answer in one turn, or ask a **clarifying question** first
and answer on a later turn. A SESSION-level judge reads the whole conversation via `{context}`
and scores the **final** answer once — the correct granularity for a multi-turn agent (a
clarify-then-answer conversation is judged on its destination, not on each intermediate turn).

So SESSION-level is chosen for **granularity**, not to dodge per-turn span gaps: answer quality
for a multi-turn conversation is a once-per-conversation question, and `{expected_response}` is
a TRACE-only placeholder (the expected answer is threaded into `{assertions}` for the SESSION
`FinalAnswerFaithfulness` judge). **Trade-off:** the per-turn `Builtin.Correctness` /
`Builtin.Helpfulness` signal is not collected; the per-turn trajectory table (clarified /
has_sql / latency) covers per-turn behaviour instead.